# PDF を Chroma に登録する

[導入ノートブックの「実習の準備」](8300.1001.RAG.introduction.ipynb)に従って依存パッケージをインストールし、Jupyter を起動する環境で `OPENAI_API_KEY` を設定します。先に実習用の PDF を確認します。

In [1]:
import hashlib
import os
import shutil
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

/tmp/ipykernel_25302/539663773.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## 処理を関数として定義する

In [2]:
def make_doc_id(source_name: str, page: int, chunk_index: int) -> str:
    raw = f"{source_name}::page={page}::chunk={chunk_index}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

In [3]:
def build_chroma_db(
    pdf_path: str,
    db_dir: str = "chroma_db",
    collection_name: str = "pdf_rag",
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
    embedding_model: str = "text-embedding-3-small",
    overwrite: bool = False,
) -> None:
    pdf_path = Path(pdf_path).resolve()
    db_dir = Path(db_dir).resolve()

    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    if os.getenv("OPENAI_API_KEY") is None:
        raise RuntimeError("OPENAI_API_KEY is not set")

    if db_dir.exists() and overwrite:
        print(f"Removing existing DB: {db_dir}")
        shutil.rmtree(db_dir)

    source_name = pdf_path.name

    print(f"Loading PDF: {pdf_path}")
    loader = PyPDFLoader(str(pdf_path))
    docs = loader.load()

    print(f"Loaded pages: {len(docs)}")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    splits = splitter.split_documents(docs)
    print(f"Created chunks: {len(splits)}")

    ids = []
    page_chunk_counter = {}

    for doc in splits:
        page = doc.metadata.get("page", 0)

        key = (source_name, page)
        chunk_index = page_chunk_counter.get(key, 0)
        page_chunk_counter[key] = chunk_index + 1

        doc_id = make_doc_id(source_name, page, chunk_index)
        ids.append(doc_id)

        doc.metadata["source"] = source_name
        doc.metadata["source_path"] = str(pdf_path)
        doc.metadata["page"] = page
        doc.metadata["chunk_index"] = chunk_index
        doc.metadata["doc_id"] = doc_id

    embeddings = OpenAIEmbeddings(model=embedding_model)

    vectorstore = Chroma(
        persist_directory=str(db_dir),
        embedding_function=embeddings,
        collection_name=collection_name,
    )

    existing = vectorstore.get(ids=ids)
    existing_ids = set(existing.get("ids", []))

    new_docs = []
    new_ids = []

    for doc, doc_id in zip(splits, ids):
        if doc_id not in existing_ids:
            new_docs.append(doc)
            new_ids.append(doc_id)

    print(f"Existing chunks skipped: {len(existing_ids)}")
    print(f"New chunks to add: {len(new_docs)}")

    if new_docs:
        vectorstore.add_documents(
            documents=new_docs,
            ids=new_ids,
        )

    print("Done.")
    print(f"DB directory: {db_dir}")

## 実習の入力と設定を確認する

In [4]:
pdf_dir = Path("examples/ARiSE_SPReAD/pdf")
pdf_paths = [pdf_dir / "arise_youryo_5.20.pdf", pdf_dir / "SPReAD_youryo.pdf"]
db_dir = Path("chroma_db")
collection_name = "pdf_rag"
embedding_model = "text-embedding-3-small"

for pdf_path in pdf_paths:
    if not pdf_path.is_file():
        raise FileNotFoundError(pdf_path)
    print(pdf_path)

examples/ARiSE_SPReAD/pdf/arise_youryo_5.20.pdf
examples/ARiSE_SPReAD/pdf/SPReAD_youryo.pdf


## PDF を分割してベクトルを保存する

各 PDF をチャンクに分け、埋め込みと出典を同じ Chroma コレクションに保存します。再実行時は登録済みのチャンク ID をスキップします。この処理では OpenAI Embeddings API を使います。

In [5]:
if os.getenv("OPENAI_API_KEY"):
    for pdf_path in pdf_paths:
        build_chroma_db(
            pdf_path=pdf_path,
            db_dir=db_dir,
            collection_name=collection_name,
            chunk_size=1000,
            chunk_overlap=200,
            embedding_model=embedding_model,
            overwrite=False,
        )
else:
    print("OPENAI_API_KEY を設定してから、このセルを実行してください。")

Loading PDF: /home/kino/kino/mi_tutorial/JupyterNotebook/mi_python_tutorial_rev1.15.0.public/8300.RAG/examples/ARiSE_SPReAD/pdf/arise_youryo_5.20.pdf
Loaded pages: 105
Created chunks: 177
Existing chunks skipped: 0
New chunks to add: 177
Done.
DB directory: /home/kino/kino/mi_tutorial/JupyterNotebook/mi_python_tutorial_rev1.15.0.public/8300.RAG/chroma_db
Loading PDF: /home/kino/kino/mi_tutorial/JupyterNotebook/mi_python_tutorial_rev1.15.0.public/8300.RAG/examples/ARiSE_SPReAD/pdf/SPReAD_youryo.pdf
Loaded pages: 50
Created chunks: 101
Existing chunks skipped: 0
New chunks to add: 101
Done.
DB directory: /home/kino/kino/mi_tutorial/JupyterNotebook/mi_python_tutorial_rev1.15.0.public/8300.RAG/chroma_db


登録が完了したら、[文書を検索して回答を生成する](8300.1201.RAG.ask_rag.ipynb)に進みます。